# Marketplace Seller Intelligence & Analytics

## 01 — Data Audit

This notebook performs the initial audit of the Olist e-commerce dataset.

Objectives:
- Understand the structure of each dataset
- Inspect columns and data types
- Identify missing values
- Check duplicates
- Understand date ranges
- Validate relationships between tables

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
# Load datasets

customers = pd.read_csv("../data/raw/CUSTOMERS.csv")
geo_location = pd.read_csv("../data/raw/GEO_LOCATION.csv")
orders = pd.read_csv("../data/raw/ORDERS.csv")
order_items = pd.read_csv("../data/raw/ORDER_ITEMS.csv")
order_payments = pd.read_csv("../data/raw/ORDER_PAYMENTS.csv")
order_reviews = pd.read_csv("../data/raw/ORDER_REVIEW_RATINGS.csv")
products = pd.read_csv("../data/raw/PRODUCTS.csv")
sellers = pd.read_csv("../data/raw/SELLERS.csv")

print("All datasets loaded successfully.")

All datasets loaded successfully.


In [3]:
# Dataset overview

datasets = {
    "Customers": customers,
    "Geo Location": geo_location,
    "Orders": orders,
    "Order Items": order_items,
    "Order Payments": order_payments,
    "Order Reviews": order_reviews,
    "Products": products,
    "Sellers": sellers
}

overview = pd.DataFrame({
    "Dataset": datasets.keys(),
    "Rows": [df.shape[0] for df in datasets.values()],
    "Columns": [df.shape[1] for df in datasets.values()]
})

overview

,Dataset,Rows,Columns
0,Customers,99441,5
1,Geo Location,19015,5
2,Orders,99441,8
3,Order Items,112650,7
4,Order Payments,103886,5
5,Order Reviews,100000,5
6,Products,32951,9
7,Sellers,3095,4


In [4]:
# Data types summary

dtype_summary = []

for name, df in datasets.items():
    dtype_summary.append({
        "Dataset": name,
        "Columns": df.shape[1],
        "Numeric Columns": df.select_dtypes(include="number").shape[1],
        "Text Columns": df.select_dtypes(include="object").shape[1],
        "Date Columns": df.select_dtypes(include="datetime").shape[1]
    })

dtype_summary = pd.DataFrame(dtype_summary)

dtype_summary

C:\Users\admin\AppData\Local\Temp\ipykernel_2464\2537489922.py:10: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  "Text Columns": df.select_dtypes(include="object").shape[1],
C:\Users\admin\AppData\Local\Temp\ipykernel_2464\2537489922.py:10: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_

,Dataset,Columns,Numeric Columns,Text Columns,Date Columns
0,Customers,5,1,4,0
1,Geo Location,5,3,2,0
2,Orders,8,0,8,0
3,Order Items,7,3,4,0
4,Order Payments,5,3,2,0
5,Order Reviews,5,1,4,0
6,Products,9,7,2,0
7,Sellers,4,1,3,0


In [5]:
# Missing values audit

missing_summary = []

for name, df in datasets.items():
    missing_count = df.isnull().sum().sum()
    total_cells = df.size
    missing_percentage = (missing_count / total_cells) * 100

    missing_summary.append({
        "Dataset": name,
        "Missing Values": missing_count,
        "Missing %": round(missing_percentage, 2)
    })

missing_summary = pd.DataFrame(missing_summary)

missing_summary

,Dataset,Missing Values,Missing %
0,Customers,0,0.00
1,Geo Location,0,0.00
2,Orders,4908,0.62
3,Order Items,0,0.00
4,Order Payments,0,0.00
5,Order Reviews,0,0.00
6,Products,2461,0.83
7,Sellers,114,0.92


In [6]:
# Detailed missing values by column

for name, df in datasets.items():
    missing = df.isnull().sum()
    missing = missing[missing > 0]

    if len(missing) > 0:
        print(f"\n{name}")
        print("-" * 40)
        print(missing)


Orders
----------------------------------------
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
dtype: int64

Products
----------------------------------------
product_category_name         623
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                2
product_length_cm               2
product_height_cm               2
product_width_cm                2
dtype: int64

Sellers
----------------------------------------
seller_city     57
seller_state    57
dtype: int64


In [7]:
# Duplicate records audit

duplicate_summary = []

for name, df in datasets.items():
    duplicate_count = df.duplicated().sum()

    duplicate_summary.append({
        "Dataset": name,
        "Duplicate Rows": duplicate_count
    })

duplicate_summary = pd.DataFrame(duplicate_summary)

duplicate_summary

,Dataset,Duplicate Rows
0,Customers,0
1,Geo Location,0
2,Orders,0
3,Order Items,0
4,Order Payments,0
5,Order Reviews,0
6,Products,0
7,Sellers,0


In [8]:
# Unique values audit

unique_summary = []

for name, df in datasets.items():
    for column in df.columns:
        unique_summary.append({
            "Dataset": name,
            "Column": column,
            "Unique Values": df[column].nunique(),
            "Missing Values": df[column].isnull().sum()
        })

unique_summary = pd.DataFrame(unique_summary)

unique_summary

,Dataset,Column,Unique Values,Missing Values
0,Customers,customer_id,99441,0
1,Customers,customer_unique_id,96096,0
2,Customers,customer_zip_code_prefix,14994,0
3,Customers,customer_city,4119,0
4,Customers,customer_state,20,0
5,Geo Location,geolocation_zip_code_prefix,19015,0
6,Geo Location,geolocation_lat,18987,0
7,Geo Location,geolocation_lng,18988,0
8,Geo Location,geolocation_city,3809,0
9,Geo Location,geolocation_state,20,0


In [9]:
# Key categorical values

print("ORDER STATUS")
print(orders["order_status"].value_counts())

print("\n" + "=" * 50)
print("PAYMENT TYPES")
print(order_payments["payment_type"].value_counts())

print("\n" + "=" * 50)
print("REVIEW SCORES")
print(order_reviews["review_score"].value_counts().sort_index())

print("\n" + "=" * 50)
print("CUSTOMER STATES")
print(customers["customer_state"].value_counts())

print("\n" + "=" * 50)
print("SELLER STATES")
print(sellers["seller_state"].value_counts())

ORDER STATUS
order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64

PAYMENT TYPES
payment_type
credit_card    76795
UPI            19784
voucher         5775
debit_card      1529
not_defined        3
Name: count, dtype: int64

REVIEW SCORES
review_score
1    11858
2     3235
3     8287
4    19200
5    57420
Name: count, dtype: int64

CUSTOMER STATES
customer_state
Andhra Pradesh       60688
Gujarat               7467
Chhattisgarh          6355
Karnataka             3866
Delhi                 3332
Madhya Pradesh        2853
Haryana               2202
Tamil Nadu            2122
Maharashtra           2005
Jammu & Kashmir       1537
Kerala                1531
Arunachal Pradesh     1119
Uttar Pradesh         1033
Rajasthan              808
Himachal Pradesh       743
Punjab                 679
Orissa                 625
West Bengal          

In [10]:
# Identify columns containing date/time information

date_candidates = []

for name, df in datasets.items():
    for column in df.columns:
        if "date" in column.lower() or "time" in column.lower():
            date_candidates.append({
                "Dataset": name,
                "Column": column,
                "Current Data Type": str(df[column].dtype)
            })

date_candidates = pd.DataFrame(date_candidates)

date_candidates

,Dataset,Column,Current Data Type
0,Orders,order_purchase_timestamp,str
1,Orders,order_delivered_carrier_date,str
2,Orders,order_delivered_customer_date,str
3,Orders,order_estimated_delivery_date,str
4,Order Items,shipping_limit_date,str
5,Order Reviews,review_creation_date,str
6,Order Reviews,review_answer_timestamp,str


In [11]:
# Validate date columns

date_columns = {
    "Orders": [
        "order_purchase_timestamp",
        "order_delivered_carrier_date",
        "order_delivered_customer_date",
        "order_estimated_delivery_date"
    ],
    "Order Items": [
        "shipping_limit_date"
    ],
    "Order Reviews": [
        "review_creation_date",
        "review_answer_timestamp"
    ]
}

date_validation = []

for dataset_name, columns in date_columns.items():
    df = datasets[dataset_name]

    for column in columns:
        converted = pd.to_datetime(df[column], errors="coerce")
        
        invalid_count = converted.isna().sum() - df[column].isna().sum()
        
        date_validation.append({
            "Dataset": dataset_name,
            "Column": column,
            "Invalid Dates": invalid_count,
            "Missing Dates": df[column].isna().sum(),
            "Min Date": converted.min(),
            "Max Date": converted.max()
        })

date_validation = pd.DataFrame(date_validation)

date_validation

,Dataset,Column,Invalid Dates,Missing Dates,Min Date,Max Date
0,Orders,order_purchase_timestamp,0,0,2016-09-04 21:15:00,2018-10-17 17:30:00
1,Orders,order_delivered_carrier_date,0,1783,2016-10-08 10:34:00,2018-09-11 19:48:00
2,Orders,order_delivered_customer_date,0,2965,2016-10-11 13:46:00,2018-10-17 13:22:00
3,Orders,order_estimated_delivery_date,0,0,2016-09-30 00:00:00,2018-11-12 00:00:00
4,Order Items,shipping_limit_date,0,0,2016-09-19 00:15:00,2020-04-09 22:35:00
5,Order Reviews,review_creation_date,0,0,2016-10-02 00:00:00,2018-08-31 00:00:00
6,Order Reviews,review_answer_timestamp,0,0,2016-10-07 18:32:00,2018-10-29 12:27:00


In [12]:
# Convert date columns to datetime

for dataset_name, columns in date_columns.items():
    for column in columns:
        datasets[dataset_name][column] = pd.to_datetime(
            datasets[dataset_name][column],
            errors="coerce"
        )

print("Date columns converted successfully.")

Date columns converted successfully.


In [13]:
# Verify date data types

for dataset_name, columns in date_columns.items():
    for column in columns:
        print(
            f"{dataset_name} → {column}: "
            f"{datasets[dataset_name][column].dtype}"
        )

Orders → order_purchase_timestamp: datetime64[us]
Orders → order_delivered_carrier_date: datetime64[us]
Orders → order_delivered_customer_date: datetime64[us]
Orders → order_estimated_delivery_date: datetime64[us]
Order Items → shipping_limit_date: datetime64[us]
Order Reviews → review_creation_date: datetime64[us]
Order Reviews → review_answer_timestamp: datetime64[us]


In [14]:
# Key integrity audit

key_checks = []

# Orders → Customers
key_checks.append({
    "Relationship": "Orders → Customers",
    "Key": "customer_id",
    "Unmatched": (~orders["customer_id"].isin(customers["customer_id"])).sum()
})

# Order Items → Orders
key_checks.append({
    "Relationship": "Order Items → Orders",
    "Key": "order_id",
    "Unmatched": (~order_items["order_id"].isin(orders["order_id"])).sum()
})

# Order Items → Sellers
key_checks.append({
    "Relationship": "Order Items → Sellers",
    "Key": "seller_id",
    "Unmatched": (~order_items["seller_id"].isin(sellers["seller_id"])).sum()
})

# Order Payments → Orders
key_checks.append({
    "Relationship": "Order Payments → Orders",
    "Key": "order_id",
    "Unmatched": (~order_payments["order_id"].isin(orders["order_id"])).sum()
})

# Order Reviews → Orders
key_checks.append({
    "Relationship": "Order Reviews → Orders",
    "Key": "order_id",
    "Unmatched": (~order_reviews["order_id"].isin(orders["order_id"])).sum()
})

# Products → Order Items
key_checks.append({
    "Relationship": "Order Items → Products",
    "Key": "product_id",
    "Unmatched": (~order_items["product_id"].isin(products["product_id"])).sum()
})

key_integrity = pd.DataFrame(key_checks)

key_integrity

,Relationship,Key,Unmatched
0,Orders → Customers,customer_id,0
1,Order Items → Orders,order_id,0
2,Order Items → Sellers,seller_id,0
3,Order Payments → Orders,order_id,0
4,Order Reviews → Orders,order_id,0
5,Order Items → Products,product_id,0


In [15]:
# Primary key uniqueness audit

primary_keys = {
    "Customers": "customer_id",
    "Orders": "order_id",
    "Products": "product_id",
    "Sellers": "seller_id"
}

pk_summary = []

for name, key in primary_keys.items():
    df = datasets[name]

    total_rows = len(df)
    unique_keys = df[key].nunique()
    duplicate_keys = total_rows - unique_keys

    pk_summary.append({
        "Dataset": name,
        "Primary Key": key,
        "Rows": total_rows,
        "Unique Keys": unique_keys,
        "Duplicate Keys": duplicate_keys
    })

pk_summary = pd.DataFrame(pk_summary)

pk_summary

,Dataset,Primary Key,Rows,Unique Keys,Duplicate Keys
0,Customers,customer_id,99441,99441,0
1,Orders,order_id,99441,99441,0
2,Products,product_id,32951,32951,0
3,Sellers,seller_id,3095,3095,0


In [16]:
# Transaction relationship audit

relationship_summary = {
    "Orders": {
        "Total Orders": orders["order_id"].nunique(),
        "Orders with Multiple Items": (
            order_items.groupby("order_id").size() > 1
        ).sum()
    },
    
    "Order Items": {
        "Total Items": len(order_items),
        "Unique Orders": order_items["order_id"].nunique(),
        "Unique Products": order_items["product_id"].nunique(),
        "Unique Sellers": order_items["seller_id"].nunique()
    },
    
    "Payments": {
        "Total Payment Records": len(order_payments),
        "Unique Orders": order_payments["order_id"].nunique()
    },
    
    "Reviews": {
        "Total Review Records": len(order_reviews),
        "Unique Orders": order_reviews["order_id"].nunique()
    }
}

relationship_summary

{'Orders': {'Total Orders': 99441,
  'Orders with Multiple Items': np.int64(9803)},
 'Order Items': {'Total Items': 112650,
  'Unique Orders': 98666,
  'Unique Products': 32951,
  'Unique Sellers': 3095},
 'Payments': {'Total Payment Records': 103886, 'Unique Orders': 99440},
 'Reviews': {'Total Review Records': 100000, 'Unique Orders': 99441}}

In [17]:
# Detailed transaction relationship analysis

# Orders without order items
orders_without_items = set(orders["order_id"]) - set(order_items["order_id"])

# Orders without payments
orders_without_payments = set(orders["order_id"]) - set(order_payments["order_id"])

# Orders with multiple payments
multiple_payment_orders = (
    order_payments.groupby("order_id").size()
    .loc[lambda x: x > 1]
)

# Orders with multiple reviews
multiple_review_orders = (
    order_reviews.groupby("order_id").size()
    .loc[lambda x: x > 1]
)

print("Orders without items:", len(orders_without_items))
print("Orders without payments:", len(orders_without_payments))
print("Orders with multiple payments:", len(multiple_payment_orders))
print("Orders with multiple reviews:", len(multiple_review_orders))

Orders without items: 775
Orders without payments: 1
Orders with multiple payments: 2961
Orders with multiple reviews: 555


In [18]:
# Analyze orders without item records

orders_no_items = orders[
    orders["order_id"].isin(orders_without_items)
]

print("Orders without items by status:")
print(orders_no_items["order_status"].value_counts())

print("\nTotal orders without items:", len(orders_no_items))

Orders without items by status:
order_status
unavailable    603
canceled       164
created          5
invoiced         2
shipped          1
Name: count, dtype: int64

Total orders without items: 775


In [19]:
# Monetary data audit

print("ORDER ITEMS PRICE SUMMARY")
print(order_items["price"].describe())

print("\n" + "=" * 50)
print("ORDER ITEMS FREIGHT VALUE SUMMARY")
print(order_items["freight_value"].describe())

print("\n" + "=" * 50)
print("PAYMENT VALUE SUMMARY")
print(order_payments["payment_value"].describe())

print("\n" + "=" * 50)
print("PAYMENT TYPES")
print(order_payments["payment_type"].value_counts())

ORDER ITEMS PRICE SUMMARY
count    112650.000000
mean        120.653739
std         183.633928
min           0.850000
25%          39.900000
50%          74.990000
75%         134.900000
max        6735.000000
Name: price, dtype: float64

ORDER ITEMS FREIGHT VALUE SUMMARY
count    112650.000000
mean         19.990320
std          15.806405
min           0.000000
25%          13.080000
50%          16.260000
75%          21.150000
max         409.680000
Name: freight_value, dtype: float64

PAYMENT VALUE SUMMARY
count    103886.000000
mean        154.100380
std         217.494064
min           0.000000
25%          56.790000
50%         100.000000
75%         171.837500
max       13664.080000
Name: payment_value, dtype: float64

PAYMENT TYPES
payment_type
credit_card    76795
UPI            19784
voucher         5775
debit_card      1529
not_defined        3
Name: count, dtype: int64


In [20]:
# Payment value audit

print(order_payments["payment_value"].describe())

print("\n" + "=" * 50)
print("Payment types:")
print(order_payments["payment_type"].value_counts())

count    103886.000000
mean        154.100380
std         217.494064
min           0.000000
25%          56.790000
50%         100.000000
75%         171.837500
max       13664.080000
Name: payment_value, dtype: float64

Payment types:
payment_type
credit_card    76795
UPI            19784
voucher         5775
debit_card      1529
not_defined        3
Name: count, dtype: int64


In [21]:
# Order-level revenue reconciliation

# Calculate total item value and freight for each order
item_totals = (
    order_items
    .groupby("order_id")
    .agg(
        item_value=("price", "sum"),
        freight_value=("freight_value", "sum")
    )
    .reset_index()
)

# Calculate total payment value for each order
payment_totals = (
    order_payments
    .groupby("order_id")
    .agg(
        payment_value=("payment_value", "sum")
    )
    .reset_index()
)

# Combine the two
revenue_check = item_totals.merge(
    payment_totals,
    on="order_id",
    how="outer"
)

# Calculate difference
revenue_check["difference"] = (
    revenue_check["payment_value"]
    - (revenue_check["item_value"] + revenue_check["freight_value"])
)

revenue_check.head()

,order_id,item_value,freight_value,payment_value,difference
0,00010242fe8c5a6d1ba2dd792cb16214,58.90,13.29,72.19,0.000000e+00
1,00018f77f2f0320c557190d7a144bdd3,239.90,19.93,259.83,0.000000e+00
2,000229ec398224ef6ca0657da4fc703e,199.00,17.87,216.87,0.000000e+00
3,00024acbcdf0a6daa1e931b038114c75,12.99,12.79,25.78,0.000000e+00
4,00042b26cf59d7ce69dfabb4e55b4fd9,199.90,18.14,218.04,-2.842171e-14


In [22]:
# Check revenue reconciliation across all orders

revenue_check["abs_difference"] = revenue_check["difference"].abs()

print("Orders checked:", len(revenue_check))

print("Exact matches:", (revenue_check["abs_difference"] < 0.01).sum())

print("Differences >= R$0.01:",
      (revenue_check["abs_difference"] >= 0.01).sum())

print("Maximum absolute difference:",
      revenue_check["abs_difference"].max())

Orders checked: 99441
Exact matches: 98284
Differences >= R$0.01: 381
Maximum absolute difference: 182.80999999999995


In [23]:
# Investigate orders with significant revenue differences

mismatches = revenue_check[
    revenue_check["abs_difference"] >= 0.01
].copy()

# Add order status
mismatches = mismatches.merge(
    orders[["order_id", "order_status"]],
    on="order_id",
    how="left"
)

# Show largest differences
mismatches.sort_values(
    "abs_difference",
    ascending=False
)[[
    "order_id",
    "item_value",
    "freight_value",
    "payment_value",
    "difference",
    "order_status"
]].head(20)

,order_id,item_value,freight_value,payment_value,difference,order_status
315,ce6d150fb29ada17d2082f4847107665,1299.00,104.66,1586.47,182.81,delivered
167,6e5fe7366a2e1bfbf3257dba0af1267f,179.19,108.72,406.92,119.01,delivered
174,70b742795bc441e94a44a084b6d9ce7a,269.99,196.94,578.82,111.89,delivered
236,996c7e73600ad3723e8627ab7bef81e4,559.90,28.00,664.43,76.53,delivered
175,70b7e94ea46d3e8b5bc12a50186edaf0,167.88,45.27,274.84,61.69,delivered
285,bc2c82b0ef78d2252b6176d1972db7c9,165.00,77.01,303.02,61.01,delivered
269,af9ffff2ce6b3defd34fd4c78857a379,395.65,17.52,466.97,53.80,delivered
54,262118ce178bb3e4590a3adcf6d62e6b,119.80,57.94,126.12,-51.62,delivered
291,bfdb5bbb06458d600a33d61f5f287472,297.00,51.93,394.36,45.43,delivered
212,8d9c0dc8d5a2ce804f6b925d8f8e6c1d,209.80,44.65,293.89,39.44,delivered


In [24]:
# Inspect the largest revenue mismatch

order_id_check = "ce6d150fb29ada17d2082f4847107665"

print("ORDER")
print(orders[orders["order_id"] == order_id_check])

print("\nORDER ITEMS")
print(order_items[order_items["order_id"] == order_id_check])

print("\nPAYMENTS")
print(order_payments[order_payments["order_id"] == order_id_check])

ORDER
                               order_id                       customer_id  \
11791  ce6d150fb29ada17d2082f4847107665  d8acb46c718975948423dbf1b603b959   

      order_status order_purchase_timestamp order_approved_at  \
11791    delivered      2018-01-25 11:17:00   1/27/2018 12:35   

      order_delivered_carrier_date order_delivered_customer_date  \
11791          2018-01-29 18:32:00           2018-02-21 17:38:00   

      order_estimated_delivery_date  
11791                    2018-03-08  

ORDER ITEMS
                               order_id  order_item_id  \
90875  ce6d150fb29ada17d2082f4847107665              1   

                             product_id                         seller_id  \
90875  638bbb2a5e4f360b71f332ddfebfd672  c4af86330efa7a2620772227d2d670c9   

      shipping_limit_date   price  freight_value  
90875 2018-02-07 12:31:00  1299.0         104.66  

PAYMENTS
                             order_id  payment_sequential payment_type  \
359  ce6d150fb29ada17d20

In [25]:
# Direction of payment differences

print("Payment greater than item + freight:",
      (revenue_check["difference"] > 0.01).sum())

print("Payment lower than item + freight:",
      (revenue_check["difference"] < -0.01).sum())

print("Within R$0.01:",
      (revenue_check["abs_difference"] <= 0.01).sum())

Payment greater than item + freight: 291
Payment lower than item + freight: 90
Within R$0.01: 98284


In [26]:
# Build seller transaction dataset

seller_transactions = order_items.merge(
    orders[
        [
            "order_id",
            "order_status",
            "order_purchase_timestamp",
            "order_delivered_customer_date",
            "order_estimated_delivery_date"
        ]
    ],
    on="order_id",
    how="left"
)

print("Seller transaction dataset created.")
print("Rows:", len(seller_transactions))
print("Columns:", len(seller_transactions.columns))

seller_transactions.head()

Seller transaction dataset created.
Rows: 112650
Columns: 11


,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,order_status,order_purchase_timestamp,order_delivered_customer_date,order_estimated_delivery_date
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:00,58.90,13.29,delivered,2017-09-13 08:59:00,2017-09-20 23:43:00,2017-09-29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:00,239.90,19.93,delivered,2017-04-26 10:53:00,2017-05-12 16:04:00,2017-05-15
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:00,199.00,17.87,delivered,2018-01-14 14:33:00,2018-01-22 13:19:00,2018-02-05
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:00,12.99,12.79,delivered,2018-08-08 10:00:00,2018-08-14 13:32:00,2018-08-20
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:00,199.90,18.14,delivered,2017-02-04 13:57:00,2017-03-01 16:42:00,2017-03-17


In [27]:
# Add seller information

seller_transactions = seller_transactions.merge(
    sellers[
        [
            "seller_id",
            "seller_zip_code_prefix",
            "seller_city",
            "seller_state"
        ]
    ],
    on="seller_id",
    how="left"
)

print("Seller information added.")
print("Rows:", len(seller_transactions))
print("Columns:", len(seller_transactions.columns))

seller_transactions.head()

Seller information added.
Rows: 112650
Columns: 14


,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,order_status,order_purchase_timestamp,order_delivered_customer_date,order_estimated_delivery_date,seller_zip_code_prefix,seller_city,seller_state
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:00,58.90,13.29,delivered,2017-09-13 08:59:00,2017-09-20 23:43:00,2017-09-29,27277,Serilingampalle,Andhra Pradesh
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:00,239.90,19.93,delivered,2017-04-26 10:53:00,2017-05-12 16:04:00,2017-05-15,3471,Akkarampalle,Andhra Pradesh
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:00,199.00,17.87,delivered,2018-01-14 14:33:00,2018-01-22 13:19:00,2018-02-05,37564,Goredi Chancha,Rajasthan
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:00,12.99,12.79,delivered,2018-08-08 10:00:00,2018-08-14 13:32:00,2018-08-20,14403,Adilabad,Andhra Pradesh
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:00,199.90,18.14,delivered,2017-02-04 13:57:00,2017-03-01 16:42:00,2017-03-17,87900,Kumhari,Chhattisgarh


In [28]:
# Add product information

seller_transactions = seller_transactions.merge(
    products[
        [
            "product_id",
            "product_category_name",
            "product_weight_g",
            "product_length_cm",
            "product_height_cm",
            "product_width_cm"
        ]
    ],
    on="product_id",
    how="left"
)

print("Product information added.")
print("Rows:", len(seller_transactions))
print("Columns:", len(seller_transactions.columns))

seller_transactions.head()

Product information added.
Rows: 112650
Columns: 19


,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,order_status,order_purchase_timestamp,order_delivered_customer_date,order_estimated_delivery_date,seller_zip_code_prefix,seller_city,seller_state,product_category_name,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:00,58.90,13.29,delivered,2017-09-13 08:59:00,2017-09-20 23:43:00,2017-09-29,27277,Serilingampalle,Andhra Pradesh,Cool_Stuff,650.0,28.0,9.0,14.0
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:00,239.90,19.93,delivered,2017-04-26 10:53:00,2017-05-12 16:04:00,2017-05-15,3471,Akkarampalle,Andhra Pradesh,Pet_Shop,30000.0,50.0,30.0,40.0
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:00,199.00,17.87,delivered,2018-01-14 14:33:00,2018-01-22 13:19:00,2018-02-05,37564,Goredi Chancha,Rajasthan,Furniture_Decor,3050.0,33.0,13.0,33.0
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:00,12.99,12.79,delivered,2018-08-08 10:00:00,2018-08-14 13:32:00,2018-08-20,14403,Adilabad,Andhra Pradesh,Perfumery,200.0,16.0,10.0,15.0
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:00,199.90,18.14,delivered,2017-02-04 13:57:00,2017-03-01 16:42:00,2017-03-17,87900,Kumhari,Chhattisgarh,Garden_Tools,3750.0,35.0,40.0,30.0


In [29]:
# Create sales value metrics

seller_transactions["gmv"] = seller_transactions["price"]

seller_transactions["total_value"] = (
    seller_transactions["price"]
    + seller_transactions["freight_value"]
)

seller_transactions[
    ["price", "freight_value", "gmv", "total_value"]
].head()


,price,freight_value,gmv,total_value
0,58.90,13.29,58.90,72.19
1,239.90,19.93,239.90,259.83
2,199.00,17.87,199.00,216.87
3,12.99,12.79,12.99,25.78
4,199.90,18.14,199.90,218.04


In [30]:
# Check analytical dataset quality

print("Total rows:", len(seller_transactions))

print("\nMissing seller IDs:")
print(seller_transactions["seller_id"].isna().sum())

print("\nMissing product categories:")
print(seller_transactions["product_category_name"].isna().sum())

print("\nMissing order status:")
print(seller_transactions["order_status"].isna().sum())

Total rows: 112650

Missing seller IDs:
0

Missing product categories:
1627

Missing order status:
0


In [31]:
print("Missing order status:", seller_transactions["order_status"].isna().sum())

Missing order status: 0


In [32]:
# Handle missing product categories

seller_transactions["product_category_name"] = (
    seller_transactions["product_category_name"]
    .fillna("Unknown")
)

print(
    "Missing product categories:",
    seller_transactions["product_category_name"].isna().sum()
)

print(
    "Total categories:",
    seller_transactions["product_category_name"].nunique()
)

Missing product categories: 0
Total categories: 72


In [33]:
# Create seller-level KPI table

seller_kpis = (
    seller_transactions
    .groupby("seller_id")
    .agg(
        total_orders=("order_id", "nunique"),
        items_sold=("order_item_id", "count"),
        gmv=("gmv", "sum"),
        total_freight=("freight_value", "sum"),
        avg_item_price=("price", "mean")
    )
    .reset_index()
)

seller_kpis.head()

,seller_id,total_orders,items_sold,gmv,total_freight,avg_item_price
0,0015a82c2db000af6aaaf3ae2ecb0532,3,3,2685.00,63.06,895.000000
1,001cca7ae9ae17fb1caed9dfb1094831,200,239,25080.03,8854.14,104.937364
2,001e6ad469a905060d959994f1b41e4f,1,1,250.00,17.94,250.000000
3,002100f778ceb8431b7a1020ff7ab48f,51,55,1234.50,793.66,22.445455
4,003554e2dce176b5555353e4f3555ac8,1,1,120.00,19.38,120.000000


In [34]:
# Calculate Average Order Value (AOV)

seller_kpis["aov"] = (
    seller_kpis["gmv"] / seller_kpis["total_orders"]
)

seller_kpis.head()

,seller_id,total_orders,items_sold,gmv,total_freight,avg_item_price,aov
0,0015a82c2db000af6aaaf3ae2ecb0532,3,3,2685.00,63.06,895.000000,895.000000
1,001cca7ae9ae17fb1caed9dfb1094831,200,239,25080.03,8854.14,104.937364,125.400150
2,001e6ad469a905060d959994f1b41e4f,1,1,250.00,17.94,250.000000,250.000000
3,002100f778ceb8431b7a1020ff7ab48f,51,55,1234.50,793.66,22.445455,24.205882
4,003554e2dce176b5555353e4f3555ac8,1,1,120.00,19.38,120.000000,120.000000


In [35]:
# Calculate delivery performance

delivered_orders = seller_transactions[
    seller_transactions["order_status"] == "delivered"
].copy()

delivered_orders["delivery_days"] = (
    delivered_orders["order_delivered_customer_date"]
    - delivered_orders["order_purchase_timestamp"]
).dt.total_seconds() / (24 * 60 * 60)

delivered_orders["delivery_delay_days"] = (
    delivered_orders["order_delivered_customer_date"]
    - delivered_orders["order_estimated_delivery_date"]
).dt.total_seconds() / (24 * 60 * 60)

print("Delivered transaction rows:", len(delivered_orders))

delivered_orders[
    ["order_id", "delivery_days", "delivery_delay_days"]
].head()

Delivered transaction rows: 110197


,order_id,delivery_days,delivery_delay_days
0,00010242fe8c5a6d1ba2dd792cb16214,7.613889,-8.011806
1,00018f77f2f0320c557190d7a144bdd3,16.215972,-2.330556
2,000229ec398224ef6ca0657da4fc703e,7.948611,-13.445139
3,00024acbcdf0a6daa1e931b038114c75,6.147222,-5.436111
4,00042b26cf59d7ce69dfabb4e55b4fd9,25.114583,-15.304167


In [36]:
# Create one row per delivered order for delivery analysis

order_delivery = (
    delivered_orders[
        [
            "order_id",
            "order_status",
            "order_purchase_timestamp",
            "order_delivered_customer_date",
            "order_estimated_delivery_date",
            "delivery_days",
            "delivery_delay_days"
        ]
    ]
    .drop_duplicates("order_id")
)

print("Unique delivered orders:", len(order_delivery))

order_delivery.head()

Unique delivered orders: 96478


,order_id,order_status,order_purchase_timestamp,order_delivered_customer_date,order_estimated_delivery_date,delivery_days,delivery_delay_days
0,00010242fe8c5a6d1ba2dd792cb16214,delivered,2017-09-13 08:59:00,2017-09-20 23:43:00,2017-09-29,7.613889,-8.011806
1,00018f77f2f0320c557190d7a144bdd3,delivered,2017-04-26 10:53:00,2017-05-12 16:04:00,2017-05-15,16.215972,-2.330556
2,000229ec398224ef6ca0657da4fc703e,delivered,2018-01-14 14:33:00,2018-01-22 13:19:00,2018-02-05,7.948611,-13.445139
3,00024acbcdf0a6daa1e931b038114c75,delivered,2018-08-08 10:00:00,2018-08-14 13:32:00,2018-08-20,6.147222,-5.436111
4,00042b26cf59d7ce69dfabb4e55b4fd9,delivered,2017-02-04 13:57:00,2017-03-01 16:42:00,2017-03-17,25.114583,-15.304167


In [37]:
# Overall delivery performance

print("Average delivery time:",
      round(order_delivery["delivery_days"].mean(), 2), "days")

print("Median delivery time:",
      round(order_delivery["delivery_days"].median(), 2), "days")

print("Average delivery delay:",
      round(order_delivery["delivery_delay_days"].mean(), 2), "days")

print("Orders delivered early:",
      (order_delivery["delivery_delay_days"] < 0).sum())

print("Orders delivered late:",
      (order_delivery["delivery_delay_days"] > 0).sum())

print("Orders delivered on time:",
      (order_delivery["delivery_delay_days"] == 0).sum())

Average delivery time: 12.56 days
Median delivery time: 10.22 days
Average delivery delay: -11.18 days
Orders delivered early: 88644
Orders delivered late: 7826
Orders delivered on time: 0


In [38]:
# Add delivery metrics to seller transactions

seller_transactions = seller_transactions.merge(
    order_delivery[
        [
            "order_id",
            "delivery_days",
            "delivery_delay_days"
        ]
    ],
    on="order_id",
    how="left"
)

print("Delivery metrics added successfully.")
print("Rows:", len(seller_transactions))

Delivery metrics added successfully.
Rows: 112650


In [39]:
seller_transactions[
    ["order_id", "seller_id", "delivery_days", "delivery_delay_days"]
].head()

,order_id,seller_id,delivery_days,delivery_delay_days
0,00010242fe8c5a6d1ba2dd792cb16214,48436dade18ac8b2bce089ec2a041202,7.613889,-8.011806
1,00018f77f2f0320c557190d7a144bdd3,dd7ddc04e1b6c2c614352b383efe2d36,16.215972,-2.330556
2,000229ec398224ef6ca0657da4fc703e,5b51032eddd242adc84c38acab88f23d,7.948611,-13.445139
3,00024acbcdf0a6daa1e931b038114c75,9d7a1d34a5052409006425275ba1c2b4,6.147222,-5.436111
4,00042b26cf59d7ce69dfabb4e55b4fd9,df560393f3a51e74553ab94004ba5c87,25.114583,-15.304167


In [40]:
# Seller-level delivery performance

seller_delivery = (
    seller_transactions[
        seller_transactions["order_status"] == "delivered"
    ][
        [
            "seller_id",
            "order_id",
            "delivery_days",
            "delivery_delay_days"
        ]
    ]
    .drop_duplicates(["seller_id", "order_id"])
)

seller_delivery_kpis = (
    seller_delivery
    .groupby("seller_id")
    .agg(
        delivered_orders=("order_id", "nunique"),
        avg_delivery_days=("delivery_days", "mean"),
        avg_delivery_delay_days=("delivery_delay_days", "mean"),
        late_orders=("delivery_delay_days", lambda x: (x > 0).sum())
    )
    .reset_index()
)

seller_delivery_kpis["late_delivery_rate"] = (
    seller_delivery_kpis["late_orders"]
    / seller_delivery_kpis["delivered_orders"]
    * 100
)

seller_delivery_kpis.head()

,seller_id,delivered_orders,avg_delivery_days,avg_delivery_delay_days,late_orders,late_delivery_rate
0,0015a82c2db000af6aaaf3ae2ecb0532,3,10.793750,-15.593750,0,0.000000
1,001cca7ae9ae17fb1caed9dfb1094831,195,13.405787,-12.225976,13,6.666667
2,002100f778ceb8431b7a1020ff7ab48f,50,16.386708,-7.390264,9,18.000000
3,003554e2dce176b5555353e4f3555ac8,1,4.646528,-26.067361,0,0.000000
4,004c9cd9d87a3c30c522c48c4fc07416,156,14.811280,-11.063555,13,8.333333


In [41]:
# Combine seller sales and delivery KPIs

seller_intelligence = seller_kpis.merge(
    seller_delivery_kpis,
    on="seller_id",
    how="left"
)

seller_intelligence.head()

,seller_id,total_orders,items_sold,gmv,total_freight,avg_item_price,aov,delivered_orders,avg_delivery_days,avg_delivery_delay_days,late_orders,late_delivery_rate
0,0015a82c2db000af6aaaf3ae2ecb0532,3,3,2685.00,63.06,895.000000,895.000000,3.0,10.793750,-15.593750,0.0,0.000000
1,001cca7ae9ae17fb1caed9dfb1094831,200,239,25080.03,8854.14,104.937364,125.400150,195.0,13.405787,-12.225976,13.0,6.666667
2,001e6ad469a905060d959994f1b41e4f,1,1,250.00,17.94,250.000000,250.000000,NaN,NaN,NaN,NaN,NaN
3,002100f778ceb8431b7a1020ff7ab48f,51,55,1234.50,793.66,22.445455,24.205882,50.0,16.386708,-7.390264,9.0,18.000000
4,003554e2dce176b5555353e4f3555ac8,1,1,120.00,19.38,120.000000,120.000000,1.0,4.646528,-26.067361,0.0,0.000000


In [42]:
# Review summary

print("Total review records:", len(order_reviews))
print("Unique orders with reviews:", order_reviews["order_id"].nunique())

print("\nReview score distribution:")
print(order_reviews["review_score"].value_counts().sort_index())

print("\nOrders with multiple reviews:",
      order_reviews.groupby("order_id").size().gt(1).sum())

Total review records: 100000
Unique orders with reviews: 99441

Review score distribution:
review_score
1    11858
2     3235
3     8287
4    19200
5    57420
Name: count, dtype: int64

Orders with multiple reviews: 555


In [43]:
# Create one review rating per order

order_reviews_clean = (
    order_reviews
    .groupby("order_id")
    .agg(
        review_score=("review_score", "mean")
    )
    .reset_index()
)

print("Unique orders with review ratings:", len(order_reviews_clean))

order_reviews_clean.head()

Unique orders with review ratings: 99441


,order_id,review_score
0,00010242fe8c5a6d1ba2dd792cb16214,5.0
1,00018f77f2f0320c557190d7a144bdd3,4.0
2,000229ec398224ef6ca0657da4fc703e,5.0
3,00024acbcdf0a6daa1e931b038114c75,4.0
4,00042b26cf59d7ce69dfabb4e55b4fd9,5.0


In [44]:
# Add review scores to seller transactions

seller_transactions = seller_transactions.merge(
    order_reviews_clean,
    on="order_id",
    how="left"
)

print("Rows:", len(seller_transactions))
print("Missing review scores:", seller_transactions["review_score"].isna().sum())

Rows: 112650
Missing review scores: 0


In [45]:
# Seller-level review performance

seller_reviews = (
    seller_transactions[
        [
            "seller_id",
            "order_id",
            "review_score"
        ]
    ]
    .drop_duplicates(["seller_id", "order_id"])
)

seller_review_kpis = (
    seller_reviews
    .groupby("seller_id")
    .agg(
        reviewed_orders=("order_id", "nunique"),
        avg_review_score=("review_score", "mean"),
        low_rating_orders=("review_score", lambda x: (x <= 2).sum())
    )
    .reset_index()
)

seller_review_kpis["low_rating_rate"] = (
    seller_review_kpis["low_rating_orders"]
    / seller_review_kpis["reviewed_orders"]
    * 100
)

seller_review_kpis.head()

,seller_id,reviewed_orders,avg_review_score,low_rating_orders,low_rating_rate
0,0015a82c2db000af6aaaf3ae2ecb0532,3,3.666667,1,33.333333
1,001cca7ae9ae17fb1caed9dfb1094831,200,3.950000,34,17.000000
2,001e6ad469a905060d959994f1b41e4f,1,1.000000,1,100.000000
3,002100f778ceb8431b7a1020ff7ab48f,51,3.901961,8,15.686275
4,003554e2dce176b5555353e4f3555ac8,1,5.000000,0,0.000000


In [46]:
# Add review performance to seller intelligence

seller_intelligence = seller_intelligence.merge(
    seller_review_kpis,
    on="seller_id",
    how="left"
)

seller_intelligence.head()

,seller_id,total_orders,items_sold,gmv,total_freight,avg_item_price,aov,delivered_orders,avg_delivery_days,avg_delivery_delay_days,late_orders,late_delivery_rate,reviewed_orders,avg_review_score,low_rating_orders,low_rating_rate
0,0015a82c2db000af6aaaf3ae2ecb0532,3,3,2685.00,63.06,895.000000,895.000000,3.0,10.793750,-15.593750,0.0,0.000000,3,3.666667,1,33.333333
1,001cca7ae9ae17fb1caed9dfb1094831,200,239,25080.03,8854.14,104.937364,125.400150,195.0,13.405787,-12.225976,13.0,6.666667,200,3.950000,34,17.000000
2,001e6ad469a905060d959994f1b41e4f,1,1,250.00,17.94,250.000000,250.000000,NaN,NaN,NaN,NaN,NaN,1,1.000000,1,100.000000
3,002100f778ceb8431b7a1020ff7ab48f,51,55,1234.50,793.66,22.445455,24.205882,50.0,16.386708,-7.390264,9.0,18.000000,51,3.901961,8,15.686275
4,003554e2dce176b5555353e4f3555ac8,1,1,120.00,19.38,120.000000,120.000000,1.0,4.646528,-26.067361,0.0,0.000000,1,5.000000,0,0.000000
